# MVA Hackathon 2026 — Reproducible Evidence Chain
**Team bigbag** · Track 1 + Track 2

Public-API-only reproduction of the evidence chain behind our BUB1B compound-heterozygous call and the Track-2 repurposing tiers. No gated genomic data is used or required — every query below hits a public database.

**Confirmed result (Track 1):** BUB1B `chr15:40209701 T>G` (p.Leu737Ter, ClinVar P/LP) + `chr15:40220612 T>G` (p.Asn1002Lys, gnomAD singleton). Score: 100.0 rank points / F-max 1.000.

## 1. The causal pair — ClinVar and gnomAD (public)

In [ ]:
# ClinVar record for allele 1 (rs759242053)
import urllib.request, json
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=clinvar&retmode=json&id=533901"
with urllib.request.urlopen(url, timeout=60) as r:
    d = json.load(r)["result"]["533901"]
print(d["title"])
print("Class:", d["germline_classification"]["description"])
print("Location:", d["variation_set"][0]["variation_loc"][0])

In [ ]:
# gnomAD v4 frequencies for both alleles
Q = '''query { variant(variantId: "%s", dataset: gnomad_r4) {
  variantId rsids genome { ac an af } exome { ac an af } } }'''
for vid in ["15-40209701-T-G", "15-40220612-T-G"]:
    res = gql("https://gnomad.broadinstitute.org/api", Q % vid)
    v = res["data"]["variant"]
    g, e = v["genome"] or {}, v["exome"] or {}
    print(vid, "rsIDs:", v["rsids"], "| genome AF", g.get("af"), "| exome AF", e.get("af"))

## 2. Consequence — Ensembl VEP (public REST)

In [ ]:
body = json.dumps({"variants": ["15 40209701 . T G . . .", "15 40220612 . T G . . ."]}).encode()
req = urllib.request.Request("https://rest.ensembl.org/vep/homo_sapiens/region", data=body,
                             headers={"Content-Type": "application/json", "Accept": "application/json"})
with urllib.request.urlopen(req, timeout=90) as r:
    res = json.load(r)
for entry in res:
    for t in entry.get("transcript_consequences", []):
        if t.get("canonical") and t.get("gene_symbol") == "BUB1B":
            print(entry["input"], "->", t["consequence_terms"], t.get("hgvsp"))

## 3. Open Targets — BUB1B profile (Track 2 anchor)

In [ ]:
Q = '''query { target(ensemblId: "ENSG00000156970") {
  approvedSymbol functionDescriptions
  tractability { label modality }
  drugAndClinicalCandidates { rows { maxClinicalStage drug { name } } } } }'''
res = gql("https://api.platform.opentargets.org/api/v4/graphql", Q)
t = res["data"]["target"]
print("Symbol:", t["approvedSymbol"])
print("Function:", t["functionDescriptions"][0][:200])
print("Tractability labels:", sorted({x["label"] for x in t["tractability"]}))
print("Registered drugs targeting BUB1B:", len(t["drugAndClinicalCandidates"]["rows"]), "(zero = the unmet need)")

## 4. DGIdb — approved levers at MTOR, WEE1, AURKA

In [ ]:
Q = '''query { genes(names: ["MTOR","WEE1","AURKA"]) {
  edges { node { name interactions { drug { name approved } } } } } }'''
res = gql("https://dgidb.org/api/graphql", Q)
for e in res["data"]["genes"]["edges"]:
    g = e["node"]
    approved = sorted({i["drug"]["name"] for i in g["interactions"] if i["drug"]["approved"]})
    print(g["name"], "approved drugs:", approved[:8])

## 5. everycure/matrix-scores — global repurposing prior (39.5M pairs)

Local scan of the full public dataset (Hugging Face). Selected results for the proband's indications:

| Indication (MONDO) | Top-ranked drugs | Reading |
|---|---|---|
| Rhabdomyosarcoma (0005212) | docetaxel, cisplatin, etoposide | standard cytotoxics — nothing new |
| MVA (0000141) | vincristine, flutamide, torasemide | vincristine = standard RMS anchor — triangulation |
| MVA1/BUB1B (0009759) | acetazolamide first | renal-axis discussion point, E4 |

Full table: `track2/evidence/matrix_scores_matches.csv` in the repo. **No strong computational prior exists for MVA — the unmet need, quantified.**

## 6. Scoring simulator — our ladder vs the challenge evaluator

The repo ships `track1/analysis/submission_sim.py`, which imports the challenge's own `evaluation.py` logic. Simulated scores for the submitted 5-row file:

- truth = submitted pair → **100.0 rank points, F-max 1.000** (actual leaderboard result)
- alternate-truth scenario → 50.0 via hedge rows

## 7. Structural mechanism — UniProt domain map (public REST)

In [ ]:
import urllib.request, json
with urllib.request.urlopen("https://rest.uniprot.org/uniprotkb/O60566.json", timeout=60) as r:
    d = json.load(r)
for f in d["features"]:
    if f["type"] in ("Domain", "Region", "Motif", "Active site", "Binding site"):
        loc = f["location"]
        print(f"{f['type']:13} {loc['start'].get('value')}-{loc['end'].get('value')}  {f.get('description', '')[:60]}")
print()
print("p.Leu737Ter: kinase-domain-specific null (scaffold 1-737 retained)")
print("p.Asn1002Lys: kinase C-lobe, 19.8 A from catalytic D882 (AlphaFold v6, pLDDT 91)")

---
*Everything above runs on free tiers from a browser (Colab) in under two minutes. Gated genomic data was used for the original analysis only, under the hackathon data terms, and will be deleted within 30 days of close.*